In [10]:
import pyarrow.parquet as pq
import os
import duckdb

In [11]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = os.path.join(base_path, "filtered")

materials_file = os.path.join(base_path, "materials_single_line.parquet")
filtered_columns_file = os.path.join(output_dir, "filtered_materials.parquet")
final_output_file = os.path.join(output_dir, "filtered_materials_encoded_time_cut.parquet")

## Step 1: Filter columns with PyArrow

In [3]:
# Columns to retain
keep_columns = [
    "component_position", "component_id", "serial_number_id",
    "station_id", "supplier_id", "mounting_place",
    "container_number", "panel_position", "created_at", "book_state"
]

In [4]:
# Load the full Parquet file
table = pq.read_table(materials_file)

# Drop columns not in the keep list
filtered_table = table.select(keep_columns)

In [5]:
# Save the filtered table first (without encoding yet)
pq.write_table(filtered_table, filtered_columns_file)
print(f"Filtered materials saved to: {filtered_columns_file}")

Filtered materials saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_materials.parquet


 ## Step 2: Encode the container_number column, fill in mounting_place null values and cut out only the desired time frame

In [12]:
con = duckdb.connect()

Define target time window, this is the timewindow of the bookings and measurements merged dataset.

In [13]:
# TODO Adjust or automate to match the timeframe of bookings and measurements
start_time = "2025-03-01T01:21:20.773Z"
end_time = "2025-05-14T01:02:06.180Z"

In [14]:
# Load the parquet, fill NULL mounting_place, frequency encode container_number
con.execute(f"""
CREATE OR REPLACE TABLE materials AS
SELECT *,
       COUNT(*) OVER (PARTITION BY container_number) AS container_number_freq
FROM (
    SELECT
        component_position,
        component_id,
        serial_number_id,
        station_id,
        supplier_id,
        COALESCE(mounting_place, 'unknown') AS mounting_place,
        container_number,
        panel_position,
        created_at,
        book_state
    FROM '{filtered_columns_file}'
    WHERE book_state != 2
);
""")

In [15]:
bookstate_counts = con.execute("""
    SELECT book_state, COUNT(*) as count
    FROM materials
    GROUP BY book_state
    ORDER BY book_state;
""").fetchdf()

print("\nBook states in 'materials' after excluding 2:")
print(bookstate_counts)


Book states in 'materials' after excluding 2:
   book_state     count
0           0  51523790
1           1       515


In [16]:
# Filter by time window
con.execute("""
    CREATE OR REPLACE TABLE materials_filtered_timewindow AS
    SELECT
        component_position, component_id, serial_number_id,
        station_id, supplier_id, mounting_place,
        panel_position, created_at, book_state, container_number_freq
    FROM materials
    WHERE created_at BETWEEN ? AND ?;
""", [start_time, end_time])

In [17]:
# Count how many failed rows are available (book_state = 1)
failed_count = con.execute("""
    SELECT COUNT(*) FROM materials_filtered_timewindow WHERE book_state = 1;
""").fetchone()[0]
print(f"\nAvailable failed rows (book_state = 1): {failed_count}")


Available failed rows (book_state = 1): 463


In [18]:
# Create balanced sample
con.execute(f"""
    CREATE OR REPLACE TABLE filtered_materials_final AS
    (
        SELECT * FROM materials_filtered_timewindow WHERE book_state = 1
    )
    UNION ALL
    (
        SELECT * FROM materials_filtered_timewindow WHERE book_state = 0 ORDER BY RANDOM() LIMIT {failed_count}
    );
""")

In [19]:
# Save to parquet
con.execute(f"COPY filtered_materials_final TO '{final_output_file}' (FORMAT 'parquet', COMPRESSION 'zstd');")
print(f"\nFinal balanced materials saved to: {final_output_file}")


Final balanced materials saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_materials_encoded_time_cut.parquet


In [20]:
bookstate_counts = con.execute("""
    SELECT book_state, COUNT(*) as count
    FROM filtered_materials_final
    GROUP BY book_state
    ORDER BY book_state;
""").fetchdf()

print("\nBook states in final balanced dataset:")
print(bookstate_counts)


Book states in final balanced dataset:
   book_state  count
0           0    463
1           1    463


In [21]:
con.close()